In [1]:
import numpy as np
import networkx as nx
import random
import matplotlib.pyplot as plt
import warnings
from tqdm import tqdm
import pandas as pd

# --- IMPORTS FOR ML ---
from sklearn.linear_model import LinearRegression

# --- USER PATH ---
from RGG_Library import RGGBuilder

# --------------------------------------------------------------------------------
# CONFIGURATION
# --------------------------------------------------------------------------------

SPACE                       = "torus"
USE_ANGLES                  = [True, False][0]
USE_GEOMETRY                = [True, False][0]
CONNECTIVITY_REGIME         = ["sc","c"][0]

N_NODES                     = 1024
K_NEIGHBORS                 = 20
N_SAMPLES                   = 1000
N_BINS                      = 20

N_TRAIN_GRAPHS              = 25
N_TEST_GRAPHS               = 25
N_GRAPHS                    = N_TRAIN_GRAPHS + N_TEST_GRAPHS # Total pool to sample from
N_SHUFFLES                  = 30  # Number of train/test splits to calculate std dev
BASE_SEED                   = 0


# --------------------------------------------------------------------------------
# RGG BUILDER
# --------------------------------------------------------------------------------

def build_RGG(iteration_seed):
    """
    Generates a graph, returns samples.
    """
    builder = RGGBuilder(
        n                           = N_NODES,
        k                           = K_NEIGHBORS,
        connectivity_regime         = CONNECTIVITY_REGIME,
        space                       = SPACE,
        perturb                     = False,
        seed                        = iteration_seed
    )
    
    G = builder.build()

    # Ensure Giant Component
    if not nx.is_connected(G):
        components = list(nx.connected_components(G))
        if not components:
            return None, None, None, None
        Gsub = G.subgraph(max(components, key=len)).copy()
    else:
        Gsub = G
    
    Gsub = nx.convert_node_labels_to_integers(Gsub, ordering="sorted")

    # Calculate Mean Degree: (2 * edges) / nodes
    mean_degree = 2 * Gsub.number_of_edges() / Gsub.number_of_nodes()

    # --- 2. Sample Data ---
    random.seed(iteration_seed)
    np.random.seed(iteration_seed)

    min_distances = builder.radius
    
    if USE_ANGLES:
        res, degs, dists, pairs, angles = RGGBuilder.sample_commute_times_even_distance_w_angles(
            Gsub, nsamples=N_SAMPLES, n_bins=N_BINS, seed=iteration_seed, 
            min_dist=min_distances, max_dist=2
        )
    else:
        res, degs, dists, pairs = RGGBuilder.sample_commute_times_even_distance(
            Gsub, nsamples=N_SAMPLES, n_bins=N_BINS, seed=iteration_seed, 
            min_dist=min_distances, max_dist=2
        )
        angles = np.zeros_like(dists)

    # --- 3. Feature Engineering ---
    safe_dists = dists.copy()
    safe_dists[safe_dists == 0] = 1e-9 

    if USE_GEOMETRY == True:
        feature_dict = {
            "Log(d)": np.log(safe_dists),
            "d^2": dists**2,
            "d^4 * cos(4theta)": dists**4 * np.cos(4*angles),
            "InvDegSum": degs,
        }
    else:
        feature_dict = {
            "InvDegSum": degs
        }

    feature_names = list(feature_dict.keys())
    X = np.column_stack(list(feature_dict.values()))
    y = np.array(res)

    return X, y, feature_names, mean_degree

# --------------------------------------------------------------------------------
# COMPUTING R^2 WITH STANDARD DEVIATIONS
# --------------------------------------------------------------------------------

def compute_r2_with_std(base_seed=BASE_SEED):
    """
    Pre-generates N_GRAPHS, then performs N_SHUFFLES random train/test splits.
    Returns a dictionary of means and standard deviations for R^2 and all coefficients.
    """
    # --- 1. Pre-generate all graphs to save time ---
    graphs_X, graphs_y, all_mean_degrees = [], [], []
    feature_names_global = None

    print(f"Building {N_GRAPHS} graphs...")
    for i in tqdm(range(N_GRAPHS)):
        X, y, feature_names, mean_degree = build_RGG(base_seed + i)
        if X is not None:
            graphs_X.append(X)
            graphs_y.append(y)
            all_mean_degrees.append(mean_degree)
            if feature_names_global is None:
                feature_names_global = feature_names

    n_built = len(graphs_X)
    if n_built < 2:
        print("Not enough valid graphs built.")
        return np.nan

    r2_list = []
    coef_list = []

    # --- 2. Perform Shuffles ---
    print(f"Running {N_SHUFFLES} random train/test splits...")
    for shuffle_idx in tqdm(range(N_SHUFFLES)):
        rng       = np.random.default_rng(seed=base_seed * 1000 + shuffle_idx)
        idx       = rng.permutation(n_built)
        
        train_idx = idx[:N_TRAIN_GRAPHS]
        test_idx  = idx[N_TRAIN_GRAPHS:N_TRAIN_GRAPHS + N_TEST_GRAPHS]

        # Failsafe if we don't have enough graphs for a split
        if len(test_idx) == 0 or len(train_idx) == 0:
            continue

        X_train = np.vstack([graphs_X[j] for j in train_idx])
        y_train = np.concatenate([graphs_y[j] for j in train_idx])
        X_test  = np.vstack([graphs_X[j] for j in test_idx])
        y_test  = np.concatenate([graphs_y[j] for j in test_idx])

        model = LinearRegression()
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        ss_res = np.sum((y_test - y_pred) ** 2)
        ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
        r2     = 1.0 - ss_res / ss_tot

        coef_dict = dict(zip(feature_names_global, model.coef_))
        coef_dict["Intercept"]   = model.intercept_
        coef_dict["Mean Degree"] = np.mean([all_mean_degrees[j] for j in train_idx])

        r2_list.append(r2)
        coef_list.append(coef_dict)

    # --- 3. Aggregate Results ---
    if not r2_list:
        return np.nan

    final_results = {
        "R^2": np.mean(r2_list),
        "R^2_std": np.std(r2_list)
    }

    # Aggregate coefficients dynamically based on whatever features were generated
    all_keys = coef_list[0].keys()
    for key in all_keys:
        vals = [d[key] for d in coef_list]
        final_results[key] = np.mean(vals)
        final_results[key + "_std"] = np.std(vals)

    return final_results

# --------------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------------
results = compute_r2_with_std(base_seed=BASE_SEED)

print("\n--- FINAL AGGREGATED RESULTS ---")
for k, v in results.items():
    print(f"{k+':':<20} {v:.5f}")

Building 50 graphs...


 38%|███▊      | 19/50 [03:45<06:07, 11.85s/it]


KeyboardInterrupt: 

In [5]:
import numpy as np
import networkx as nx
import random
import matplotlib.pyplot as plt
import warnings
from tqdm import tqdm
import pandas as pd

# --- IMPORTS FOR ML ---
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# --- USER PATH ---
from RGG_Library import RGGBuilder

# --------------------------------------------------------------------------------
# CONFIGURATION
# --------------------------------------------------------------------------------

SPACE                       = "torus"
USE_ANGLES                  = [True, False][0]
USE_GEOMETRY                = [True, False][0]
LASSO_ALPHA                 = 1e-6
CONNECTIVITY_REGIME         = ["sc","c"][0]

N_NODES                     = 1024
K_NEIGHBORS                 = 20
N_SAMPLES                   = 1000
N_BINS                      = 20

N_TRAIN_GRAPHS              = 25
N_TEST_GRAPHS               = 25
N_GRAPHS                    = N_TRAIN_GRAPHS + N_TEST_GRAPHS # Total pool to sample from
N_SHUFFLES                  = 30  # Number of train/test splits to calculate std dev
BASE_SEED                   = 0


# --------------------------------------------------------------------------------
# RGG BUILDER
# --------------------------------------------------------------------------------

def build_RGG(iteration_seed):
    """
    Generates a graph, returns samples.
    """
    builder = RGGBuilder(
        n                           = N_NODES,
        k                           = K_NEIGHBORS,
        connectivity_regime         = CONNECTIVITY_REGIME,
        space                       = SPACE,
        perturb                     = False,
        seed                        = iteration_seed
    )
    
    G = builder.build()

    # Ensure Giant Component
    if not nx.is_connected(G):
        components = list(nx.connected_components(G))
        if not components:
            return None, None, None, None
        Gsub = G.subgraph(max(components, key=len)).copy()
    else:
        Gsub = G
    
    Gsub = nx.convert_node_labels_to_integers(Gsub, ordering="sorted")

    # Calculate Mean Degree: (2 * edges) / nodes
    mean_degree = 2 * Gsub.number_of_edges() / Gsub.number_of_nodes()

    # --- 2. Sample Data ---
    random.seed(iteration_seed)
    np.random.seed(iteration_seed)

    min_distances = builder.radius
    
    if USE_ANGLES:
        res, degs, dists, pairs, angles = RGGBuilder.sample_commute_times_even_distance_w_angles(
            Gsub, nsamples=N_SAMPLES, n_bins=N_BINS, seed=iteration_seed, 
            min_dist=min_distances, max_dist=2
        )
    else:
        res, degs, dists, pairs = RGGBuilder.sample_commute_times_even_distance(
            Gsub, nsamples=N_SAMPLES, n_bins=N_BINS, seed=iteration_seed, 
            min_dist=min_distances, max_dist=2
        )
        angles = np.zeros_like(dists)

    # --- 3. Feature Engineering ---
    safe_dists = dists.copy()
    safe_dists[safe_dists == 0] = 1e-9 

    if USE_GEOMETRY == True:
        feature_dict = {
            "Log(d)": np.log(safe_dists),
            "d^2": dists**2,
            "d^4 * cos(4theta)": dists**4 * np.cos(4*angles),
            "InvDegSum": degs,
        }
    else:
        feature_dict = {
            "InvDegSum": degs
        }

    feature_names = list(feature_dict.keys())
    X = np.column_stack(list(feature_dict.values()))
    y = np.array(res)

    return X, y, feature_names, mean_degree

# --------------------------------------------------------------------------------
# COMPUTING R^2 WITH STANDARD DEVIATIONS
# --------------------------------------------------------------------------------

def compute_r2_with_std(base_seed=BASE_SEED):
    """
    Pre-generates N_GRAPHS, then performs N_SHUFFLES random train/test splits.
    Returns a dictionary of means and standard deviations for R^2 and all coefficients.
    """
    # --- 1. Pre-generate all graphs to save time ---
    graphs_X, graphs_y, all_mean_degrees = [], [], []
    feature_names_global = None

    print(f"Building {N_GRAPHS} graphs...")
    for i in tqdm(range(N_GRAPHS)):
        X, y, feature_names, mean_degree = build_RGG(base_seed + i)
        if X is not None:
            graphs_X.append(X)
            graphs_y.append(y)
            all_mean_degrees.append(mean_degree)
            if feature_names_global is None:
                feature_names_global = feature_names

    n_built = len(graphs_X)
    if n_built < 2:
        print("Not enough valid graphs built.")
        return np.nan

    r2_list = []
    coef_list = []

    # --- 2. Perform Shuffles ---
    print(f"Running {N_SHUFFLES} random train/test splits...")
    for shuffle_idx in tqdm(range(N_SHUFFLES)):
        rng       = np.random.default_rng(seed=base_seed * 1000 + shuffle_idx)
        idx       = rng.permutation(n_built)
        
        train_idx = idx[:N_TRAIN_GRAPHS]
        test_idx  = idx[N_TRAIN_GRAPHS:N_TRAIN_GRAPHS + N_TEST_GRAPHS]

        # Failsafe if we don't have enough graphs for a split
        if len(test_idx) == 0 or len(train_idx) == 0:
            continue

        X_train = np.vstack([graphs_X[j] for j in train_idx])
        y_train = np.concatenate([graphs_y[j] for j in train_idx])
        X_test  = np.vstack([graphs_X[j] for j in test_idx])
        y_test  = np.concatenate([graphs_y[j] for j in test_idx])

        model = make_pipeline(
            StandardScaler(),
            Lasso(alpha=LASSO_ALPHA, random_state=0, max_iter=50000)
        )
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_train, y_train)

        lasso          = model.named_steps['lasso']
        scaler         = model.named_steps['standardscaler']
        real_coefs     = lasso.coef_ / scaler.scale_
        real_intercept = lasso.intercept_ - np.sum(real_coefs * scaler.mean_)

        y_pred = X_test @ real_coefs + real_intercept
        ss_res = np.sum((y_test - y_pred) ** 2)
        ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
        r2     = 1.0 - ss_res / ss_tot

        coef_dict = dict(zip(feature_names_global, real_coefs))
        coef_dict["Intercept"]   = real_intercept
        coef_dict["Mean Degree"] = np.mean([all_mean_degrees[j] for j in train_idx])

        r2_list.append(r2)
        coef_list.append(coef_dict)

    # --- 3. Aggregate Results ---
    if not r2_list:
        return np.nan

    final_results = {
        "R^2": np.mean(r2_list),
        "R^2_std": np.std(r2_list)
    }

    # Aggregate coefficients dynamically based on whatever features were generated
    all_keys = coef_list[0].keys()
    for key in all_keys:
        vals = [d[key] for d in coef_list]
        final_results[key] = np.mean(vals)
        final_results[key + "_std"] = np.std(vals)

    return final_results

# --------------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------------
results = compute_r2_with_std(base_seed=BASE_SEED)

print("\n--- FINAL AGGREGATED RESULTS ---")
for k, v in results.items():
    print(f"{k+':':<20} {v:.5f}")

Building 50 graphs...


100%|██████████| 50/50 [09:40<00:00, 11.60s/it]


Running 30 random train/test splits...


100%|██████████| 30/30 [00:00<00:00, 75.92it/s]


--- FINAL AGGREGATED RESULTS ---
R^2:                 0.95825
R^2_std:             0.00118
Log(d):              0.02477
Log(d)_std:          0.00012
d^2:                 -0.03852
d^2_std:             0.00051
d^4 * cos(4theta):   -0.01699
d^4 * cos(4theta)_std: 0.00065
InvDegSum:           1.17751
InvDegSum_std:       0.00444
Intercept:           0.05334
Intercept_std:       0.00049
Mean Degree:         19.99872
Mean Degree_std:     0.02652
